# Anforderungen an Projektumsetzung: Clustering

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt



#### Führen Sie mit dem Algorithmus Ihrer Wahl eine Clusteranalyse auf Ihren Daten durch.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Daten mit spezifischen Spalten laden
columns_to_use = ['MaxAge', 'YearsCode', 'WorkExp', 'JobSat', 'ConvertedCompYearly']
df = pd.read_csv("survey_results_shortened.csv", usecols=columns_to_use)

df_q98 = (df[df['ConvertedCompYearly'] <= df['ConvertedCompYearly'].quantile(0.98)])


# 2. Fehlende Werte behandeln
df_clean = df_q98.dropna()  # Oder geeignete Imputation

# 3. Daten skalieren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

# 4. Beste Clusteranzahl finden
K = range(2, 15)
inertias = []


for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)


# 5. Elbow- und Silhouetten-Plots
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(K, inertias, 'bx-')
plt.title('Elbow-Methode')
plt.xlabel('Anzahl Cluster (k)')
plt.ylabel('Inertia')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Daten laden
columns_to_use = ['MaxAge', 'YearsCode', 'WorkExp', 'JobSat', 'ConvertedCompYearly']
df = pd.read_csv("survey_results_shortened.csv", usecols=columns_to_use)

# Optional: Ausreißer beim Gehalt entfernen (wie bei euch)
df = df[df['ConvertedCompYearly'] <= df['ConvertedCompYearly'].quantile(0.98)]

# 2. Fehlende Werte entfernen
df_clean = df.dropna()

# 3. Skalieren (wichtig für PCA!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

# 4. PCA mit 2 Komponenten (zur Visualisierung)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 5. Varianz anzeigen
print("Erklärte Varianz pro Komponente:", pca.explained_variance_ratio_)
print("Gesamt erklärte Varianz:", pca.explained_variance_ratio_.sum())

# 6. Scatterplot der PCA-Komponenten
plt.figure(figsize=(6, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA der Umfragedaten")
plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

best_k = 5

# 7. Finales Modell
final_kmeans = KMeans(n_clusters=best_k, random_state=42)
clusters = final_kmeans.fit_predict(X_scaled)

# 8. Visualisierung mit MaxAge und WorkExp
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df_clean['WorkExp'],
                     df_clean['YearsCode'],
                     c=clusters,
                     cmap='viridis',
                     alpha=0.6)
plt.xlabel('WorkExp')
plt.ylabel('YearsCode')
plt.title(f'K-Means Clustering (k={best_k}) - MaxAge vs WorkExp')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 9. Cluster-Statistiken
df_clean['Cluster'] = clusters
print("\nCluster-Statistiken:")
print(df_clean.groupby('Cluster')[columns_to_use].mean().round(2))

In [ ]:
df = pd.read_csv("survey_results_shortened.csv")


df_q98 = (df[df['ConvertedCompYearly'] <= df['ConvertedCompYearly'].quantile(0.98)])
# df_q98.describe()
df

In [ ]:
import pandas as pd, ast
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import numpy as np

# 1. Lade Datei
df = pd.read_csv("survey_results_shortened.csv")

# 2. Spalten (anpassen falls nötig)
list_columns = [
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DevType"
]

categorical_columns = [
    "EdLevel",
    "Employment",
    "OrgSize",
    "Country",
    "RemoteWork"
]

# 3. Auswahl, parse list-strings
df_selected = df[[c for c in list_columns + categorical_columns if c in df.columns]].copy()

def parse_list(x):
    if isinstance(x, str) and x.startswith("["):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return [] if pd.isna(x) else []

for col in list_columns:
    if col in df_selected.columns:
        df_selected[col] = df_selected[col].apply(parse_list)
        df_selected[col] = df_selected[col].apply(lambda lst: [s.strip().lower() for s in lst])

# 4. Expand & One-Hot für Listen-Spalten (robust!)
expanded_df = df_selected.copy()

for col in list_columns:
    if col not in expanded_df.columns:
        continue

    # explode → dummies → wieder zusammenfassen
    exploded = expanded_df[col].explode()

    # Falls die Spalte komplett leer ist
    if exploded.dropna().empty:
        continue

    dummies = pd.get_dummies(exploded).groupby(level=0).sum()

    dummies = dummies.add_prefix(col + "_")

    # Jetzt sauber anhängen (Index passt!)
    expanded_df = pd.concat([expanded_df, dummies], axis=1)

# Original Listen entfernen
expanded_df = expanded_df.drop(columns=[c for c in list_columns if c in expanded_df.columns])


# 5. One-Hot für normale Kategorien
expanded_df = pd.get_dummies(expanded_df, columns=[c for c in categorical_columns if c in expanded_df.columns], dummy_na=True)

# 6. Feature-Matrix
X = expanded_df.fillna(0)
# pca = PCA(n_components=0.90)  # 90% der Varianz behalten
# X_reduced = pca.fit_transform(X)
#
# print("Vor PCA:", X.shape)
# print("Nach PCA:", X_reduced.shape)

# 7. Elbow: inertias berechnen
Ks = range(2, 15)
inertias = []

# Nur 30.000 Zeilen verwenden (oder weniger falls dein PC schwach ist)
# max_sample = min(30000, X_reduced.shape[0])
# X_sample = X_reduced[:max_sample]

for k in Ks:
    km = MiniBatchKMeans(
        n_clusters=k,
        batch_size=2048,
        random_state=42
    )
    km.fit(X)
    inertias.append(km.inertia_)

# 8. Plot
plt.figure(figsize=(8,5))
plt.plot(list(Ks), inertias, marker='o')
plt.xlabel("Anzahl Cluster (k)")
plt.ylabel("Inertia")
plt.title("Elbow-Plot")
plt.grid(True)
plt.show()

# 9. Wähle best_k per Plot und berechne finale Labels
best_k = 6  # <- durch Plot bestimmen
kmeans_final = MiniBatchKMeans(
    n_clusters=best_k,
    batch_size=2048,
    random_state=42
)
clusters = kmeans_final.fit_predict(X)

df["Cluster"] = clusters

# ================================
# 5. Ausgabe
# ================================
# print(df["Cluster"].value_counts())
# for c in range(6):
#     print(f"\n=== Cluster {c} ===")
#     display(df[df["Cluster"] == c].head(5))

top_n = 10
cluster_profiles = expanded_df.groupby(df["Cluster"]).mean()
cluster_profiles
for c in sorted(df["Cluster"].unique()):
    print(f"\n===== Cluster {c} — Top {top_n} Features =====")
    print(cluster_profiles.loc[c].sort_values(ascending=False).head(top_n))


In [ ]:
df = pd.read_csv("survey_results_shortened.csv")

df

In [ ]:
df = df.drop(columns=['EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence', 'ToolCountWork', 'ToolCountPersonal', 'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry', 'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry', 'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry', 'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry', 'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry', 'OpSysPersonal use', 'OpSysProfessional use', 'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr', 'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry', 'AISent', 'AIAcc', 'AIComplex', 'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task", 'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI', 'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain', 'AIAgentChange', 'AgentUsesGeneral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree', 'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree', 'AIAgentChallengesStrongly disagree', 'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration', 'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite', 'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen'])

In [ ]:
df = df.drop(columns=df.columns[df.columns.str.startswith('TechEndorse')])
df = df.drop(columns=df.columns[df.columns.str.startswith('TechOppose')])
df = df.drop(columns=df.columns[df.columns.str.startswith('JobSatPoints')])


In [ ]:
df.to_csv("survey_results_cleaned_up.csv", index=False)
